# Colab GPU work · sentence vectors and DistilRoBERTa

The laptop has no CUDA GPU, so the two heavy jobs of the project run here, on Google Colab:

1. **Sentence vectors** for every petition with `all-MiniLM-L6-v2` and `all-mpnet-base-v2`.
   They are used for duplicate retrieval (07) and for the similarity features (08).
2. **Fine-tuning `distilroberta-base`** on the train split (the improved classifier), with early stopping on
   validation macro-F1, and its validation logits (for temperature scaling and the fusion model in 08).

**Input**, in the Google Drive folder `DRIVE_DIR` (uploaded from `data/processed/`): `petitions.parquet`,
`out_of_scope.parquet`, `splits.parquet`. The code in `src/` comes from the GitHub repository.

**Output**, in `OUTPUT_DIR` on Google Drive:

| File | Content | Copy to (laptop) |
|---|---|---|
| `embeddings/minilm_vectors.npy` + `minilm_ids.npy` | 128,077 × 384 vectors of length 1, and the petition id of each row | `data/processed/embeddings/` |
| `embeddings/mpnet_vectors.npy` + `mpnet_ids.npy` | 128,077 × 768 vectors, same order rule | `data/processed/embeddings/` |
| `distilroberta/best_model/` | the fine-tuned model (best epoch) and its tokenizer | `models/distilroberta/` |
| `distilroberta/val_predictions.parquet` | `id`, `logit_<class>`, `p_<class>` for the 4,030 validation petitions | `data/processed/predictions/` |
| `distilroberta/training_log.csv` | loss and validation scores during training | `reports/` |
| `colab_environment.json` | versions, GPU, settings, minutes per step | `reports/` |

**Two modes** (`DRY_RUN` in section 1):
- `DRY_RUN = True`: a small test on a **CPU** runtime: 1,000 texts for the vectors and 35 train / 56 validation
  petitions for the fine-tuning, written to a separate `dry_run/` folder. About 15 minutes. It runs every step
  once with the real settings, so a wrong path or a changed library shows up without using GPU time.
- `DRY_RUN = False`: the real run on a **T4 GPU** runtime, about 1–1.5 hours.

**If Colab disconnects:** run all cells again. Finished steps are skipped because their files are already on Drive,
and the training continues from the last saved epoch.

**The test split is not used here.** Vectors are also computed for the test petitions, but the sentence models are
fixed: nothing is learned from those texts, and the test vectors are only looked at in notebook 09.

## 1. Settings

Change `DRY_RUN` for the real run. Change `DRIVE_DIR` if the Drive folder has another name.

In [ ]:
DRY_RUN = True    # True: small test on a CPU runtime. False: the real run on a T4 GPU runtime.

DRIVE_DIR = "/content/drive/MyDrive/capstone"    # Google Drive folder with the parquet files
OUTPUT_DIR = f"{DRIVE_DIR}/dry_run" if DRY_RUN else f"{DRIVE_DIR}/outputs"
REPO_URL = "https://github.com/YasithaNV2001/Petition-Moderation-Assistant-CS5998-Capstone-Project.git"
REPO_DIR = "/content/repo"

# Sentence models for duplicate retrieval (locked)
EMBEDDING_MODELS = {"minilm": "sentence-transformers/all-MiniLM-L6-v2",
                    "mpnet": "sentence-transformers/all-mpnet-base-v2"}

# DistilRoBERTa fine-tuning (locked settings)
MODEL_NAME = "distilbert/distilroberta-base"   # the current Hugging Face name of "distilroberta-base"
MAX_LEN = 256           # tokens per text; longer texts are cut
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3              # at most; training stops early when validation macro-F1 does not improve
PATIENCE = 1            # ... for 1 epoch, and the best epoch is kept
WARMUP_SHARE = 0.06     # the learning rate rises from 0 during the first 6% of steps (the value of the RoBERTa paper)
WEIGHT_DECAY = 0.01
SEED = 42

# Dry-run sizes
DRY_TEXTS, DRY_TRAIN_PER_CLASS, DRY_VAL_PER_CLASS = 1000, 5, 8

## 2. Google Drive, code and libraries

Colab asks for permission to open Google Drive. `src/preprocess.py` (the class order and the leakage check) is
taken from the public GitHub repository, so Colab uses exactly the committed code.

In [ ]:
import glob, json, os, platform, subprocess, sys, time
from google.colab import drive

drive.mount("/content/drive")
for name in ["petitions.parquet", "out_of_scope.parquet", "splits.parquet"]:
    path = f"{DRIVE_DIR}/{name}"
    assert os.path.exists(path), f"Not found on Google Drive: {path} (check DRIVE_DIR in section 1)"
    print(f"{name:22s} {os.path.getsize(path) / 1e6:5.1f} MB")
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f"{REPO_DIR}/src")
from preprocess import CLASSES, MODEL_INPUTS, leakage_check
print("code from GitHub commit", REPO_COMMIT, "| classes:", CLASSES)

In [ ]:
%pip install -q sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
import sentence_transformers

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "none (CPU runtime)"
if not DRY_RUN:
    assert GPU, "The real run needs a GPU: Runtime > Change runtime type > T4 GPU"

VERSIONS = {"python": platform.python_version(), "torch": torch.__version__,
            "transformers": transformers.__version__, "sentence-transformers": sentence_transformers.__version__,
            "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__}
TIMINGS = {}    # minutes per step, saved in section 6
print("mode:", "DRY RUN" if DRY_RUN else "REAL RUN", "| GPU:", GPU_NAME)
for package, version in VERSIONS.items():
    print(f"  {package:22s} {version}")

## 3. Load the data

Only three columns are read: `id`, `label` and `text`. Nothing else from the petitions reaches this notebook.
The asserts check that Drive holds the same files as the laptop (counts from 02 and 04).

In [ ]:
petitions = pd.read_parquet(f"{DRIVE_DIR}/petitions.parquet", columns=["id", "label", "text"])
out_of_scope = pd.read_parquet(f"{DRIVE_DIR}/out_of_scope.parquet", columns=["id", "label", "text"])
splits = pd.read_parquet(f"{DRIVE_DIR}/splits.parquet")
petitions = petitions.merge(splits, on="id", how="left", validate="one_to_one")

assert (len(petitions), len(out_of_scope)) == (127_191, 886)
assert petitions["split"].value_counts().to_dict() == {"train": 116_417, "test": 6_182, "val": 4_030, "excluded": 562}
assert set(petitions["label"]) == set(CLASSES)
pd.crosstab(petitions["label"], petitions["split"]).reindex(index=CLASSES, columns=["train", "val", "test", "excluded"])

## 4. Sentence vectors

Every petition gets one vector per model: the 127,191 in-scope petitions and the 886 out-of-scope ones (08 needs
their similarity features), 128,077 in total. Each vector has length 1, so the cosine similarity of two petitions
is simply the dot product of their vectors.

- The text keeps its original casing, as the models were trained on normal text.
- Texts longer than the model's limit are cut by the library (MiniLM 256, mpnet 384 word pieces; printed below).
- On the GPU the models run in half precision (float16), which is about 3× faster. The vectors are stored as
  float32 and set to length 1 again.
- Row `i` of `*_vectors.npy` belongs to the petition id in row `i` of `*_ids.npy`.

In [ ]:
texts = pd.concat([petitions[["id", "text"]], out_of_scope[["id", "text"]]], ignore_index=True)
assert texts["id"].is_unique and len(texts) == 128_077
if DRY_RUN:
    texts = texts.sample(DRY_TEXTS, random_state=SEED)
leakage_check(texts[MODEL_INPUTS])     # the sentence models see only the text
os.makedirs(f"{OUTPUT_DIR}/embeddings", exist_ok=True)
print(f"{len(texts):,} texts")

In [ ]:
from sentence_transformers import SentenceTransformer

for short_name, model_name in EMBEDDING_MODELS.items():
    vectors_path = f"{OUTPUT_DIR}/embeddings/{short_name}_vectors.npy"
    ids_path = f"{OUTPUT_DIR}/embeddings/{short_name}_ids.npy"
    if os.path.exists(ids_path):       # the ids are saved last, so they also mark a finished step
        print(f"{short_name}: already saved, skipped")
        continue
    start = time.time()
    encoder = SentenceTransformer(model_name, device=DEVICE)
    if GPU:
        encoder.half()
    vectors = encoder.encode(texts["text"].tolist(), batch_size=128, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)    # exactly length 1 again after float16
    np.save(vectors_path, vectors)
    np.save(ids_path, texts["id"].to_numpy())
    TIMINGS[f"vectors_{short_name}"] = round((time.time() - start) / 60, 1)
    print(f"{short_name}: {vectors.shape}, texts cut at {encoder.max_seq_length} word pieces, "
          f"{TIMINGS[f'vectors_{short_name}']} min")
    del encoder
    if GPU:
        torch.cuda.empty_cache()

Check the saved files. As an example, a gold pair from the data audit that shares almost no words:
#150557 "Ban the t.v licence" was rejected as a duplicate of #121973 "Stop TV licensing!". Its cosine should be
clearly higher than the average of random pairs (in the dry run these two petitions are usually not in the sample).

In [ ]:
rng = np.random.default_rng(SEED)
for short_name in EMBEDDING_MODELS:
    vectors = np.load(f"{OUTPUT_DIR}/embeddings/{short_name}_vectors.npy")
    ids = np.load(f"{OUTPUT_DIR}/embeddings/{short_name}_ids.npy")
    assert len(vectors) == len(ids) == len(texts) and set(ids) == set(texts["id"])
    assert np.isfinite(vectors).all() and np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4)
    a, b = rng.integers(0, len(ids), size=(2, 2000))
    print(f"{short_name}: {vectors.shape}, mean cosine of 2,000 random pairs {np.sum(vectors[a] * vectors[b], axis=1).mean():.2f}")
    row = {pid: i for i, pid in enumerate(ids)}
    if 150557 in row and 121973 in row:
        print(f"  cosine of #150557 and #121973: {vectors[row[150557]] @ vectors[row[121973]]:.2f}")

## 5. Fine-tune DistilRoBERTa

**What fine-tuning does:** `distilroberta-base` has already learned English from a large amount of general text.
A new layer with 7 outputs (one per class) is put on top, and the whole network is trained on the train petitions
for a few passes (epochs), so it learns which wording leads to which moderation outcome.

- **Input:** `text` only, original casing, cut at 256 tokens.
- **Class weights:** each class counts the same in total in the loss ("balanced" weights, n / (7 × n_class),
  the same rule as `class_weight="balanced"` for the SVC). Without them the model could ignore `local`
  (265 train petitions, weight ≈ 63).
- **Early stopping:** after every epoch the model is scored on validation (macro-F1). The best epoch is kept, and
  training stops when an epoch brings no improvement.
- **Checkpoints:** saved on Drive after every epoch, so a disconnect loses at most one epoch.
- **Mixed precision (fp16)** on the GPU, which is faster on a T4.
- **Seed 42.** GPU training is not exactly repeatable to the last digit, so the saved model and its logits are
  the reference results.

### 5.1 Data and class weights

In [ ]:
label2id = {c: i for i, c in enumerate(CLASSES)}    # CLASSES order, also used for the logit_ / p_ columns
train = petitions[petitions["split"] == "train"]
val = petitions[petitions["split"] == "val"]
if DRY_RUN:
    train = train.groupby("label").sample(n=DRY_TRAIN_PER_CLASS, random_state=SEED)
    val = val.groupby("label").sample(n=DRY_VAL_PER_CLASS, random_state=SEED)
leakage_check(train[MODEL_INPUTS])
leakage_check(val[MODEL_INPUTS])
y_train = train["label"].map(label2id).to_numpy()
y_val = val["label"].map(label2id).to_numpy()

counts = np.bincount(y_train, minlength=len(CLASSES))
class_weights = len(y_train) / (len(CLASSES) * counts)     # balanced: every class counts the same in total
pd.DataFrame({"train petitions": counts, "weight": class_weights.round(2)}, index=CLASSES)

### 5.2 Tokens

The tokenizer splits each text into word pieces and turns them into numbers. Padding (filling short texts up to
the longest text of the batch) is added later, per batch, which is faster than padding everything to 256.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PetitionDataset(torch.utils.data.Dataset):
    """The token ids of each text (cut at MAX_LEN) and its class number."""
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item

start = time.time()
train_ds = PetitionDataset(train["text"], y_train)
val_ds = PetitionDataset(val["text"], y_val)
lengths = np.array([len(ids) for ids in train_ds.encodings["input_ids"]])
print(f"tokenised in {time.time() - start:.0f} s; train texts: median {np.median(lengths):.0f} tokens, "
      f"{(lengths == MAX_LEN).mean():.1%} cut at {MAX_LEN}")

### 5.3 Loss and validation score

Hugging Face's `Trainer` runs the training loop. Two small additions: the loss uses the class weights, and after
every epoch it computes macro-F1 on validation, which chooses the best epoch.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

ALL_LABELS = list(range(len(CLASSES)))
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    """The normal Trainer, with class-weighted cross-entropy as the loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels,
                                                 weight=weights_tensor.to(labels.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predicted = eval_pred.predictions.argmax(axis=1)
    return {"macro_f1": f1_score(eval_pred.label_ids, predicted, average="macro", labels=ALL_LABELS, zero_division=0),
            "accuracy": accuracy_score(eval_pred.label_ids, predicted)}

### 5.4 Training

The longest step: about 12–15 minutes per epoch on a T4. Each epoch ends with a validation score and a checkpoint
on Drive. `save_total_limit=1` keeps only the best checkpoint, plus the latest one if it is a different epoch (to
continue after a disconnect).

In [ ]:
CHECKPOINT_DIR = f"{OUTPUT_DIR}/distilroberta/checkpoints"
BEST_DIR = f"{OUTPUT_DIR}/distilroberta/best_model"
LOG_PATH = f"{OUTPUT_DIR}/distilroberta/training_log.csv"    # written last, so it marks a finished training

if os.path.exists(LOG_PATH):
    print("already trained, skipped; the model is in", BEST_DIR)
else:
    set_seed(SEED)    # also fixes the random start values of the new 7-output layer
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=label2id)
    steps_per_epoch = int(np.ceil(len(train_ds) / BATCH_SIZE))
    args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=LEARNING_RATE,
        warmup_steps=int(WARMUP_SHARE * steps_per_epoch * EPOCHS),
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=GPU,
        logging_steps=1 if DRY_RUN else 200,
        report_to="none",
        seed=SEED,
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer), processing_class=tokenizer,
        compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

    checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    if checkpoints:
        print("continuing from", checkpoints[-1])
    start = time.time()
    trainer.train(resume_from_checkpoint=checkpoints[-1] if checkpoints else None)
    TIMINGS["fine_tuning"] = round((time.time() - start) / 60, 1)

    trainer.save_model(BEST_DIR)     # the best epoch (load_best_model_at_end) and the tokenizer
    pd.DataFrame(trainer.state.log_history).to_csv(LOG_PATH, index=False)
    print(f"best: {trainer.state.best_model_checkpoint}, validation macro-F1 {trainer.state.best_metric:.4f}, "
          f"{TIMINGS['fine_tuning']} min")

### 5.5 Validation predictions

The best model is loaded from Drive and scores every validation petition. A plain loop in float32 is used (the
same function will give the test predictions on the laptop in 09). The logits are the raw scores; the
probabilities are their softmax with temperature 1. Notebook 08 fits the temperature on these logits.

In [ ]:
def predict_logits(model, texts, batch_size=64):
    """Logits (n_texts x 7, in CLASSES order) for a list of texts."""
    model.eval()
    logits = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = tokenizer(texts[start:start + batch_size], truncation=True, max_length=MAX_LEN,
                              padding=True, return_tensors="pt").to(DEVICE)
            logits.append(model(**batch).logits.float().cpu().numpy())
    return np.vstack(logits)

best_model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(DEVICE)
assert [best_model.config.id2label[i] for i in ALL_LABELS] == CLASSES
val_logits = predict_logits(best_model, val["text"].tolist())
val_probs = torch.softmax(torch.from_numpy(val_logits), dim=1).numpy()

predictions = pd.DataFrame({"id": val["id"].to_numpy()})
for i, c in enumerate(CLASSES):
    predictions[f"logit_{c}"] = val_logits[:, i]
for i, c in enumerate(CLASSES):
    predictions[f"p_{c}"] = val_probs[:, i]
predictions.to_parquet(f"{OUTPUT_DIR}/distilroberta/val_predictions.parquet", index=False)
assert set(predictions["id"]) == set(val["id"]) and np.allclose(val_probs.sum(axis=1), 1, atol=1e-5)
print(predictions.shape)

### 5.6 Validation results

DistilRoBERTa on text only, before temperature scaling and fusion. The macro-F1 below can differ very slightly from
the best epoch's score in the training log: during training the validation pass ran in float16, here in float32.

In [ ]:
log = pd.read_csv(LOG_PATH)
print(log.dropna(subset=["eval_macro_f1"])[["epoch", "eval_loss", "eval_macro_f1", "eval_accuracy"]].to_string(index=False))

predicted = val_logits.argmax(axis=1)
VAL_MACRO_F1 = f1_score(y_val, predicted, average="macro", labels=ALL_LABELS, zero_division=0)
print(f"\nvalidation macro-F1 {VAL_MACRO_F1:.4f}\n")
print(classification_report(y_val, predicted, labels=ALL_LABELS, target_names=CLASSES, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(y_val, predicted, labels=ALL_LABELS),
             index=[f"true {c}" for c in CLASSES], columns=CLASSES)

## 6. Record the environment

Versions, GPU, settings and minutes per step, for the reproducibility checklist. If an earlier session already
finished some steps, their minutes are kept from the earlier file.

In [ ]:
env_path = f"{OUTPUT_DIR}/colab_environment.json"
earlier = json.load(open(env_path)) if os.path.exists(env_path) else {}
environment = {
    "mode": "dry run" if DRY_RUN else "real run",
    "date_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "gpu": GPU_NAME,
    "versions": VERSIONS,
    "code_commit": REPO_COMMIT,
    "settings": {"model": MODEL_NAME, "embedding_models": EMBEDDING_MODELS, "max_len": MAX_LEN,
                 "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "max_epochs": EPOCHS,
                 "patience": PATIENCE, "warmup_share": WARMUP_SHARE, "weight_decay": WEIGHT_DECAY, "seed": SEED,
                 "fp16": GPU,
                 "class_weights": dict(zip(CLASSES, class_weights.round(4).tolist()))},
    "rows": {"vectors": len(texts), "train": len(train), "val": len(val)},
    "val_macro_f1": round(float(VAL_MACRO_F1), 4),
    "minutes": {**earlier.get("minutes", {}), **TIMINGS},
}
with open(env_path, "w") as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))

## 7. Files to download

**After the dry run:** check that every cell ran without an error, then delete the `dry_run` folder on Drive (it
holds about 2 GB, mostly test checkpoints) and empty the Drive bin. To repeat a dry run from the start, delete the
folder first, because finished steps are skipped.

**After the real run:** download the files in the table at the top to the laptop. The `checkpoints` folder can be
deleted once `best_model` is on the laptop. Also save this notebook with its outputs (File → Download → .ipynb)
over `notebooks/colab_gpu.ipynb`, so the repository keeps the record of the run.

In [ ]:
for folder, _, files in sorted(os.walk(OUTPUT_DIR)):
    if "checkpoints" in folder:
        continue
    for name in sorted(files):
        path = os.path.join(folder, name)
        print(f"{os.path.relpath(path, OUTPUT_DIR):55s} {os.path.getsize(path) / 1e6:8.1f} MB")